# 04: Train a first model

Can team statistics on **July 1** predict the team's win percentage over the remaining season?

We compare predicting .500, predicting the current win percentage, and linear regression. We also try adding pitching strikeout-minus-walk rate to the regression.

Run 01, 02, and 03 first. Open this notebook from the project folder using the project's `.venv` kernel. This notebook only reads the saved training Parquet file; it makes no API requests.

## 1. Import the tools and choose the inputs

`X` means the input features. `y` means the outcome we want to predict. Only the columns listed below go into the regression.

The environment needs pandas, pyarrow, and scikit-learn. If scikit-learn is missing, install it in your notebook environment with `%pip install scikit-learn`.

In [10]:
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATA_DIR = Path("data")
TARGET = "remaining_win_pct"

BASE_FEATURES = ["off_xbsr_per_game", "allowed_xbsr_per_game"]
EXTENDED_FEATURES = BASE_FEATURES + ["pitch_k_minus_bb_rate"]

## 2. Read the data

Use July 1 only so predictions happen at a similar point each season. Exclude the shortened 2020 season. Each remaining row represents one team in one season.

Final records and columns starting with `remaining_` describe future results. They are not model inputs; `remaining_win_pct` is used only as the target.

In [11]:
data = pd.read_parquet(DATA_DIR / "training_data.parquet")
cutoff_dates = pd.to_datetime(data["Cutoff_Date"])

july = data[cutoff_dates.dt.strftime("%m-%d") == "07-01"].copy()
july = july[july["Season"] != 2020]
july = july[july["Season"].between(2016, 2025)]
july = july.sort_values(["Season", "Team_ID"]).reset_index(drop=True)

assert not july.duplicated(["Season", "Team_ID"]).any()
assert not july[EXTENDED_FEATURES + [TARGET, "win_pct"]].isna().any().any()
assert july[TARGET].between(0, 1).all()

print(f"July team-season rows: {len(july)}")
july[["Season", "Team", "Cutoff_Date", "win_pct"] + BASE_FEATURES].head()

July team-season rows: 270


,Season,Team,Cutoff_Date,win_pct,off_xbsr_per_game,allowed_xbsr_per_game
0,2016,Los Angeles Angels,2016-07-01,0.400000,4.157231,4.853351
1,2016,Arizona Diamondbacks,2016-07-01,0.439024,4.842590,4.982355
2,2016,Baltimore Orioles,2016-07-01,0.594937,5.046930,4.705979
3,2016,Boston Red Sox,2016-07-01,0.544304,5.624467,4.413659
4,2016,Chicago Cubs,2016-07-01,0.645570,5.098980,3.278650


## 3. Split by season

- **Training, 2016–2022:** learn the regression coefficients.
- **Validation, 2023–2024:** choose between the two regression versions.
- **Test, 2025:** evaluate the chosen version after the choice is made.

Keep these years separate. A random split would not represent predicting a later season. We will not use 2025 to choose features.

In [12]:
train = july[july["Season"].between(2016, 2022)].copy()
validation = july[july["Season"].between(2023, 2024)].copy()
test = july[july["Season"] == 2025].copy()

# For this dataset there should be 30 teams per included season.
assert len(train) == 180
assert len(validation) == 60
assert len(test) == 30

split_summary = pd.DataFrame({
    "split": ["Training", "Validation", "Test"],
    "seasons": ["2016-2022, excluding 2020", "2023-2024", "2025"],
    "rows": [len(train), len(validation), len(test)],
})
split_summary

,split,seasons,rows
0,Training,"2016-2022, excluding 2020",180
1,Validation,2023-2024,60
2,Test,2025,30


## 4. Fit two simple regressions

The first uses estimated runs scored and allowed per game. The second adds pitching strikeouts minus walks per batter faced.

`fit` learns from training data. `predict` applies the learned relationship to other rows. Linear regression can predict outside 0–1, so we clip all regression predictions to that range before scoring. This rule is fixed before evaluating either version.

In [13]:
base_model = LinearRegression()
base_model.fit(train[BASE_FEATURES], train[TARGET])

extended_model = LinearRegression()
extended_model.fit(train[EXTENDED_FEATURES], train[TARGET])

id_columns = ["Season", "Cutoff_Date", "Team_ID", "Team", TARGET]
validation_predictions = validation[id_columns].copy()
validation_predictions["always_500"] = 0.5
validation_predictions["current_win_pct"] = validation["win_pct"]
validation_predictions["baseruns"] = base_model.predict(validation[BASE_FEATURES]).clip(0, 1)
validation_predictions["baseruns_k_bb"] = extended_model.predict(validation[EXTENDED_FEATURES]).clip(0, 1)

## 5. Compare validation errors

**Mean absolute error (MAE)** is the average distance between predicted and actual win percentage. Lower is better. An MAE of 0.050 means an average miss of **5 percentage points**. Every team-season receives equal weight.

The two simple guesses are benchmarks. The regression needs to beat them to show that it adds value.

In [14]:
validation_scores = []
for name in ["always_500", "current_win_pct", "baseruns", "baseruns_k_bb"]:
    mae = mean_absolute_error(validation_predictions[TARGET], validation_predictions[name])
    validation_scores.append({"model": name, "mae": mae, "mae_percentage_points": mae * 100})

validation_results = pd.DataFrame(validation_scores)
validation_results.sort_values("mae").reset_index(drop=True)

,model,mae,mae_percentage_points
0,baseruns_k_bb,0.058229,5.822873
1,baseruns,0.058795,5.879506
2,always_500,0.068647,6.864703
3,current_win_pct,0.075429,7.542891


## 6. Choose the regression using validation only

Choose the regression with the lower validation MAE. Keep the simpler version if there is a tie. This selects between the two regressions; the benchmarks may still be better.

After choosing the features, refit that version on training **plus validation** data. These seasons are all earlier than the test season.

In [15]:
base_mae = mean_absolute_error(validation_predictions[TARGET], validation_predictions["baseruns"])
extended_mae = mean_absolute_error(validation_predictions[TARGET], validation_predictions["baseruns_k_bb"])

if extended_mae < base_mae:
    selected_name = "baseruns_k_bb"
    selected_features = EXTENDED_FEATURES
else:
    selected_name = "baseruns"
    selected_features = BASE_FEATURES

print("Selected regression:", selected_name)
print("Selected features:", selected_features)

train_and_validation = pd.concat([train, validation], ignore_index=True)
final_model = LinearRegression()
final_model.fit(train_and_validation[selected_features], train_and_validation[TARGET])

Selected regression: baseruns_k_bb
Selected features: ['off_xbsr_per_game', 'allowed_xbsr_per_game', 'pitch_k_minus_bb_rate']


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3,)","[ 0.06,-0.07, 0.4 ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](3,)","['off_xbsr_per_game','allowed_xbsr_per_game','pitch_k_minus_bb_rate']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.4674
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(3)


## 7. Evaluate on 2025

Now evaluate the chosen regression and the two fixed benchmarks on the 30 test teams. Do not change the features based on this result and then describe another score on 2025 as an untouched test.

One season is a small test. It tells us what happened here, not how well the model will always perform.

In [16]:
test_predictions = test[id_columns].copy()
test_predictions["always_500"] = 0.5
test_predictions["current_win_pct"] = test["win_pct"]
test_predictions[selected_name] = final_model.predict(test[selected_features]).clip(0, 1)

test_scores = []
for name in ["always_500", "current_win_pct", selected_name]:
    mae = mean_absolute_error(test_predictions[TARGET], test_predictions[name])
    test_scores.append({"model": name, "mae": mae, "mae_percentage_points": mae * 100})

test_results = pd.DataFrame(test_scores)
display(test_results.sort_values("mae").reset_index(drop=True))

model_mae = mean_absolute_error(test_predictions[TARGET], test_predictions[selected_name])
record_mae = mean_absolute_error(test_predictions[TARGET], test_predictions["current_win_pct"])
constant_mae = mean_absolute_error(test_predictions[TARGET], test_predictions["always_500"])

print(f"Selected regression MAE: {model_mae * 100:.2f} percentage points")
if model_mae < min(record_mae, constant_mae):
    print("The selected regression beat both benchmarks on the 2025 test.")
else:
    print("The selected regression did not beat both benchmarks on the 2025 test.")

,model,mae,mae_percentage_points
0,baseruns_k_bb,0.067774,6.777402
1,always_500,0.069323,6.932327
2,current_win_pct,0.076656,7.665598


Selected regression MAE: 6.78 percentage points
The selected regression beat both benchmarks on the 2025 test.


## 8. Inspect the largest misses

Look at a few team-level predictions to understand the error. Investigate possible explanations without assuming these results prove a cause.

In [17]:
test_predictions["absolute_error"] = abs(test_predictions[TARGET] - test_predictions[selected_name])
test_predictions.sort_values("absolute_error", ascending=False).head(6)

,Season,Cutoff_Date,Team_ID,Team,remaining_win_pct,always_500,current_win_pct,baseruns_k_bb,absolute_error
246,2025,2025-07-01,114,Cleveland Guardians,0.607595,0.5,0.481928,0.441789,0.165806
260,2025,2025-07-01,139,Tampa Bay Rays,0.394737,0.5,0.546512,0.537827,0.143090
254,2025,2025-07-01,133,Athletics,0.540541,0.5,0.409091,0.404781,0.135759
248,2025,2025-07-01,116,Detroit Tigers,0.441558,0.5,0.623529,0.569659,0.128100
269,2025,2025-07-01,158,Milwaukee Brewers,0.641026,0.5,0.559524,0.514138,0.126887
263,2025,2025-07-01,142,Minnesota Twins,0.389610,0.5,0.470588,0.506704,0.117094


## 9. Save results as Parquet

Save the prediction rows and the comparison tables. Validation predictions came from training on 2016–2022; test predictions came from the chosen regression refitted on 2016–2024. The saved regression column name identifies which version was selected.

These are evaluation results, not live forecasts. Running the notebook again replaces these four output files.

In [19]:
validation_predictions.to_parquet(DATA_DIR / "validation_predictions.parquet", index=False)
validation_results.to_parquet(DATA_DIR / "validation_results.parquet", index=False)
test_predictions.to_parquet(DATA_DIR / "test_predictions.parquet", index=False)
test_results.to_parquet(DATA_DIR / "test_results.parquet", index=False)

# Check that the saved predictions match the comparison score.
saved_predictions = pd.read_parquet(DATA_DIR / "test_predictions.parquet")
saved_mae = abs(saved_predictions[TARGET] - saved_predictions[selected_name]).mean()
display(test_results)
display(test_predictions)
assert abs(saved_mae - model_mae) < 0.000000001
print("Saved four Parquet files. The saved predictions reproduce the test MAE.")

,model,mae,mae_percentage_points
0,always_500,0.069323,6.932327
1,current_win_pct,0.076656,7.665598
2,baseruns_k_bb,0.067774,6.777402


,Season,Cutoff_Date,Team_ID,Team,remaining_win_pct,always_500,current_win_pct,baseruns_k_bb,absolute_error
240,2025,2025-07-01,108,Los Angeles Angels,0.384615,0.5,0.500000,0.438373,0.053758
241,2025,2025-07-01,109,Arizona Diamondbacks,0.480519,0.5,0.505882,0.516372,0.035853
242,2025,2025-07-01,110,Baltimore Orioles,0.493506,0.5,0.435294,0.426388,0.067119
243,2025,2025-07-01,111,Boston Red Sox,0.613333,0.5,0.494253,0.533010,0.080323
244,2025,2025-07-01,112,Chicago Cubs,0.545455,0.5,0.588235,0.563796,0.018341
245,2025,2025-07-01,113,Cincinnati Reds,0.513158,0.5,0.511628,0.503629,0.009529
246,2025,2025-07-01,114,Cleveland Guardians,0.607595,0.5,0.481928,0.441789,0.165806
247,2025,2025-07-01,115,Colorado Rockies,0.311688,0.5,0.223529,0.349267,0.037579
248,2025,2025-07-01,116,Detroit Tigers,0.441558,0.5,0.623529,0.569659,0.128100
249,2025,2025-07-01,117,Houston Astros,0.467532,0.5,0.600000,0.546158,0.078626


Saved four Parquet files. The saved predictions reproduce the test MAE.


## What to learn from this run

Explain the difference between training, validation, and testing, and why a regression can lose to a simple guess. The added feature is useful only if evaluation supports it.

Once you have looked at 2025, use earlier seasons for further experiments or treat 2025 as a known benchmark. Later work can evaluate several chronological splits, add other cutoff months, and compare errors by month. Keep the current simple model as a reference.

References: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [mean absolute error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html), and [avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).